# Onafhankelijke E2-schaal · vijfepochpilot
## CNN-8 / SVHN · E4M3 → E2M3 → E2M1 · A32 · B32

**Onderzoeksvraag:** kan één zelfstandig geleerde absolute shared scale voor de E2-formaten de gezamenlijke training ondersteunen, zonder een vaste verhouding of een menu van tien verhoudingen tegenover E4 op te leggen?

De E4-bronschaal volgt de bestaande amax-regel. **E2M3 en E2M1 delen een tweede E8M0-schaal per groep.** De lagere elementcodes blijven uit de E4-broncodes afgeleid via exponenttruncatie en daarna mantissatruncatie. De initialisatie is numeriek gelijk aan ×64, maar die relatie wordt daarna niet behouden of opnieuw berekend.

Deze notebook leest uitsluitend opgeslagen JSON/CSV-bestanden. Zij start geen training, laadt geen modelcheckpoints en verandert geen onderzoeksdata. Grafieken worden pas opgeslagen wanneer `SAVE_FIGURES = True`; dat maakt een nieuwe map.

In [ ]:
from pathlib import Path
RUN_DIR = None  # Of: Path(r"C:\Users\renzo\master-thesis-mx\mx_runs\mx_triple_independent_e2_scale_pilot5\..._joint_run")
SAVE_FIGURES = False

In [ ]:
import importlib.util
import sys
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

package_name = "mx_triple_independent_e2_scale_pilot5_v1"
candidates = ([Path(RUN_DIR)] if RUN_DIR is not None else []) + [Path.cwd()]
for base in [Path.cwd(), *list(Path.cwd().parents)[:4]]:
    candidates.append(base / "software_checks" / package_name)
helper_path = next((p / "notebook_support.py" for p in candidates if (p / "notebook_support.py").is_file()), None)
if helper_path is None:
    raise FileNotFoundError("notebook_support.py ontbreekt. Open de notebook vanuit de pakket- of resultatenmap, of vul RUN_DIR in.")
spec = importlib.util.spec_from_file_location("mx_independent_notebook_support", helper_path)
h = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = h
spec.loader.exec_module(h)
run = h.load_run(RUN_DIR)
figures = {}
display(Markdown(h.status_markdown(run)))
display(h.diagnostics_table(run))

## 1. Wat verandert precies?

Voor groep $g$ op tijdstip $t$ is de bestaande E4-schaal $S_4(g,t)=2^{e_4(g,t)}$. De nieuwe parameter is een **absolute log₂-schaal**:

$$\phi(g,0)=e_4(g,0)+6,\qquad e_2(g,t)=\lfloor\phi(g,t)\rfloor+\mathbf{1}\!\left[\phi(g,t)-\lfloor\phi(g,t)\rfloor\geq\tfrac12\right],\qquad S_2(g,t)=2^{e_2(g,t)}.$$

De relatie met de actuele bron, $\Delta(g,t)=e_2(g,t)-e_4(g,t)$ en $S_2/S_4=2^\Delta$, is vervolgens **een geobserveerde uitkomst**. Zij is niet de getrainde parameter en is niet tot de vroegere tien keuzes begrensd. De absolute E2-schaal blijft bij een onveranderde φ gelijk, ook wanneer de E4-bronschaal verandert. De lagere **elementcodes** blijven wel van de E4-bron afhangen.

Het FP32-hulpparameterbereik is [−127,127]. De hard-forward gebruikt alleen een E8M0-schaal (geen extra mantissa). Niet elke combinatie van een hoge E8M0-exponent en een E2-elementwaarde blijft eindig in FP32; de uitvoering geeft dan een expliciete fout en clipt dit niet stilzwijgend.

**Implementatiebronnen:** `PROTOCOL.md`, `run_train.py`, `trunc_adapter.py` en `trunc_support.py` in dit pakket. Dit is een verkennende, nieuwe proefopzet; het is nog geen bewezen accuracyverbetering.

In [ ]:
display(h.protocol_table(run))

## 2. Accuracyverloop en historische vergelijkingen

Alle grafieken gebruiken de volledige beschikbare pilotgeschiedenis. De accuracy-as zoomt in op het relevante bereik en bevat **alle getoonde waarden**; eventuele lage accuracies worden dus niet verborgen.

De vaste ×64-referentie en de tienkeuzereferentie zijn eerdere, afzonderlijke trainingen. Voor alle drie de varianten worden uitsluitend dezelfde eerste epochs getoond. Er wordt geen nieuwe pilot met een epoch100-eindscore vergeleken.

In [ ]:
display(Markdown(h.comparison_markdown(run)))

In [ ]:
fig = h.plot_learning(run)
figures["01_accuracy_verloop"] = fig
display(fig)
plt.close(fig)

### Laatste epoch en selectie op beste E4M3

Links staat de laatste **gemeenschappelijke** epoch, normaal epoch5. Rechts kiest elke variant de eerste hoogste E4M3-testaccuracy binnen hetzelfde beschikbare venster (maximaal de eerste vijf epochs); E2M3 en E2M1 komen steeds uit diezelfde epoch.

Bij de nieuwe pilot verwijst de selectie naar het opgeslagen best/last-checkpoint. Bij de historische vaste ×64-run is de selectie binnen de eerste vijf epochs hier uit de metrics afgeleid; zij is **niet** het beste checkpoint van de volledige 100-epochrun. Selectie op de officiële testset maakt deze beste score geen onafhankelijke testschatting.

In [ ]:
display(Markdown("**Laatste gemeenschappelijke epoch**"))
display(h.checkpoint_table(run, "last").round(4))
display(Markdown("**Beste E4M3 binnen hetzelfde venster**"))
display(h.checkpoint_table(run, "best").round(4))
display(Markdown("**Verschil op het gemeenschappelijke eindpunt** — procentpunten en aantallen testbeelden"))
display(h.difference_table(run).round(4))

In [ ]:
fig = h.plot_checkpoints(run)
figures["02_gemeenschappelijke_checkpoints"] = fig
display(fig)
plt.close(fig)

## 3. Welke schalen ontstaan?

Deze sectie scheidt de **geleerde absolute E2-exponent** van de **geobserveerde verhouding tot de bewegende E4-bronschaal**. Een verandering van die verhouding kan zowel door E2-leren als door een gewijzigde E4-bronschaal ontstaan.

De eerste grafiek vergelijkt initialisatie (epoch0) met de laatste beschikbare epoch. Daarna volgt de verdeling per epoch. Alle waargenomen verhoudingen worden getoond, ook waarden buiten de vroegere ×1…×512. Een niet-waargenomen waarde binnen een histogram betekent nul groepen; ontbrekende schaaldata worden nooit door nul vervangen.

In [ ]:
fig = h.plot_scale_distributions(run)
figures["03_schaalverdelingen_init_eind"] = fig
display(fig)
plt.close(fig)
display(h.scale_distribution_table(run).round(4))

In [ ]:
fig = h.plot_scale_evolution(run)
figures["04_schaalverdelingen_per_epoch"] = fig
display(fig)
plt.close(fig)

### Laagverschillen

De volgende curves tonen gemiddelden over groepen, geen betrouwbaarheidsintervallen. De tabel behoudt ook minimum en maximum; een gemiddelde alleen kan tegengestelde veranderingen tussen groepen verbergen.

In [ ]:
fig = h.plot_scale_by_layer(run)
figures["05_schalen_per_laag"] = fig
display(fig)
plt.close(fig)
display(h.scale_layer_table(run).round(5))

### Schaalveranderingen en gradientdiagnostiek

Links tellen we groepen waarvan de **opgeslagen waarde tussen twee opeenvolgende epochs** veranderde. Een groep die tussendoor heen en terug wisselde kan daar onzichtbaar blijven. Rechts staan daarom ook de tijdens de updates gemeten E2-wissels, inclusief terugwissels.

Een niet-nul proxyupdate hoeft niet meteen tot een andere harde exponent te leiden. Ook het omgekeerde geldt: een kleine update dicht bij een afrondingsgrens kan een discrete schaalwissel veroorzaken. Dat is een eigenschap van deze lokale surrogate-optimalisatie, geen bewijs dat expliciet alle exponenten zijn onderzocht.

In [ ]:
fig = h.plot_scale_changes(run)
figures["06_bron_en_e2_veranderingen"] = fig
display(fig)
plt.close(fig)
display(h.source_change_table(run))

In [ ]:
fig = h.plot_scale_gradients(run)
figures["07_gradient_en_projectie"] = fig
display(fig)
plt.close(fig)

## 4. Opslagkosten: wat kost de onafhankelijke schaal?

Per groep blijft één E4M3-broncode per gewichtspositie opgeslagen. Alle modi worden daaruit afgeleid. Alleen de schaalinformatie verschilt:

- **Vaste ×64:** één E4-schaalbyte; de E2-schaal wordt afgeleid.
- **Tien relatieve keuzes:** één E4-schaalbyte + vier bits keuze-informatie.
- **Onafhankelijke E2-schaal:** één E4-schaalbyte + één absolute E2-schaalbyte. E2M3 en E2M1 delen die tweede byte.

Bij 4.080 groepen is dit **4.080 bytes extra tegenover vaste ×64** en **2.040 bytes extra tegenover tien relatieve keuzes**. Beide schaalbanken samen kosten 0,5 bit per B32-positie; de eerste E4-bank is daarin meegeteld.

De onderstaande payloadberekening geldt voor de **zes MX-lagen inclusief groepspadding**. Zij sluit FP32-randlagen, BN-banken, bestandscontainerheaders, activaties en frameworkgeheugen uit. De training behoudt 4.080 FP32-proxies; proxy + gradient + twee Adam-momenten tellen analytisch 65.280 bytes, net als bij de tienkeuzemethode. Adam-stapmetadata en tijdelijke tensors vallen daarbuiten.

Deze getallen zijn **opslagberekeningen**, geen meting van geheugenverkeer, latency of energie. Hoeveel bytes bij precisiewissels werkelijk moeten worden opgehaald hangt ook van het opslag- en cachebeleid af; deze pilot meet dat niet.

In [ ]:
display(h.storage_cost_table(run))
fig = h.plot_storage_cost(run)
figures["08_opslagkosten"] = fig
display(fig)
plt.close(fig)

### Opgeslagen modelrepresentaties

De runner controleert de afgeleide gewichten tegen dezelfde live trainingsroute. Compacte bronexports bewaren de E4-codes en absolute lagere schaalbytes; zij bevatten geen masters van de zes MX-lagen en geen FP32-schaalproxies. Hervatbare trainingscheckpoints bewaren die FP32-toestand wel.

De tabel hieronder vermeldt **geserialiseerde containerbytes**, die ook hulpstaat en bestandsstructuur bevatten. Die zijn niet gelijk aan de theoretische schaalpayload uit de voorgaande grafiek. De exacte gewichtscontrole voert geen extra volledige testset-evaluatie uit.

In [ ]:
display(h.export_table(run))

## 5. Gemeten pilotduur en uitvoerbaarheid

Alleen werkelijk afgeronde epochs tellen als meting. Een eventueel 100-epochtotaal is een projectie voor een latere nieuwe training vanaf epoch0, met het oorspronkelijke LR-schema. Dit pakket zelf stopt na maximaal vijf epochs.

In [ ]:
display(Markdown(h.timing_markdown(run)))
fig = h.plot_timing(run)
figures["09_tijd_en_gpugeheugen"] = fig
display(fig)
plt.close(fig)

## 6. Controles, herkomst en interpretatie

**Wat deze pilot kan aantonen:** technische uitvoerbaarheid, eindige gradients en optimizerstaat, behoud van de bittruncatieroute, daadwerkelijk leerbare absolute E2-schalen, en het vroege accuracyverloop.

**Wat zij niet aantoont:** betere eindaccuracy op epoch100, onafhankelijkheid van initialisatie, statistische significantie van kleine verschillen, of een hardwareversnelling. Een eigen E2-schaal betekent hier onafhankelijkheid van de actuele E4-**schaal**, terwijl de lagere elementcodes bewust uit de E4-bron blijven komen.

De codebestanden en de gelezen meetbestanden vormen de primaire bronnen voor deze notebook. Historische referenties worden via hun `REFERENCE.json` en de daarin opgenomen hashes gecontroleerd. Verschillen in de oude vaste referentie kunnen niet op identieke samplevolgorde worden nagekeken omdat de bijbehorende volledige history hier niet meegeleverd is. De onderstaande audit maakt ontbrekende of conflicterende informatie zichtbaar.

In [ ]:
display(h.audit_table(run))
display(h.reference_table(run))
display(h.diagnostics_table(run))

## 7. Optioneel: figuren bewaren

Standaard blijft deze stap uitgeschakeld. Met `SAVE_FIGURES = True` komen PNG- en PDF-bestanden plus een herkomstbestand in een nieuwe map naast de resultaten. Ruwe data en eerdere figuren worden niet overschreven.

In [ ]:
if SAVE_FIGURES:
    saved = h.save_figures(figures, run)
    print("Nieuwe figurenmap:", saved)
else:
    print("Figuren alleen weergegeven; niets opgeslagen.")